In [ ]:
from collections import Counter
import gzip
import pathlib

import torch
import pandas as pd
import scipy
import numpy as np

import scanpy as sc
import anndata as ad
import gseapy

import seaborn as sns
import matplotlib.pyplot as plt

import flipcrow
import flipcrow.paths
import flipcrow.scp1644_mothership as mship

mps_device = torch.device("mps")

# sc.settings.verbosity = 0
import warnings
warnings.filterwarnings("ignore")

sc.set_figure_params(figsize=(10, 10))
pd.options.display.width=400

%matplotlib inline

In [ ]:
def pseudobulk_leiden(single_cell_data: ad.AnnData) -> ad.AnnData:
    clust_id_buf = []
    clust_pb_buf = []
    clust_pb_n_cells_buf = []
    for clust_id in sorted(map(int, single_cell_data.obs.leiden.unique())):
        clust_id_buf.append(str(clust_id))
        clust_pb_buf.append(single_cell_data[single_cell_data.obs.leiden == str(clust_id), :].layers['trimmed_counts'].sum(axis=0))
        clust_pb_n_cells_buf.append(len(single_cell_data[single_cell_data.obs.leiden == str(clust_id), :]))
    
    adpb = ad.AnnData(pd.DataFrame(clust_pb_buf, index=clust_id_buf, columns=single_cell_data.var_names))
    adpb.obs['leiden'] = pd.Categorical(clust_id_buf)
    adpb.obs['n_cells'] = np.array(clust_pb_n_cells_buf)
    # print(adpb)
    
    adpb.var['mt'] = adpb.var_names.str.startswith(("MT-"))
    sc.pp.calculate_qc_metrics(
        adpb, 
        qc_vars=["mt"], #, "ribo", "hb"], 
        inplace=True, 
        percent_top=[20], 
        log1p=False,
    )
    
    adpb.layers['trimmed_counts'] = adpb.to_df().loc[adpb.obs_names, :]
    # sc.pp.scrublet(adpb_all)
    sc.pp.normalize_total(adpb, target_sum=10000, inplace=True)
    sc.pp.log1p(adpb, copy=False)
    
    return adpb


In [ ]:
scp1644_raw = flipcrow.scp1644_mothership.load_scp1644_data(check_metadata=True)

In [ ]:
scp1644_trim = flipcrow.scp1644_mothership.preprocess_scp1644_data(scp1644_raw)


In [ ]:
savepoint = 'scp1644_trim_savepoint.h5ad'
scp1644_trim.write_h5ad(str(flipcrow.paths.DATA_PATH / "SCP1644" / savepoint))
# scp1644_trim = ad.read_h5ad(str(flipcrow.paths.DATA_PATH / "SCP1644" / savepoint))

In [ ]:
print(scp1644_trim[scp1644_trim.obs['Coarse_Cell_Annotations'] == 'Tumor'])
print(scp1644_trim[scp1644_trim.obs['Coarse_Cell_Annotations'] != 'Tumor'])

In [ ]:
print("PCA / NN / Leiden / UMAP / tSNE")
sc.pp.pca(scp1644_trim)
sc.pp.neighbors(scp1644_trim, n_neighbors=40, n_pcs=50)
sc.tl.leiden(scp1644_trim)
sc.tl.umap(scp1644_trim)
sc.tl.tsne(scp1644_trim)

print("HVG / Rank genes groups")
sc.pp.highly_variable_genes(scp1644_trim, flavor="seurat", n_top_genes=1070)
sc.tl.rank_genes_groups(scp1644_trim, 'leiden', method='wilcoxon')

In [ ]:
adpb_trim = pseudobulk_leiden(scp1644_trim)

In [ ]:
# clust_id_buf = []
# clust_pb_buf = []
# clust_pb_n_cells_buf = []
# for clust_id in sorted(map(int, scp1644_trim.obs.leiden.unique())):
#     clust_id_buf.append(clust_id)
#     clust_pb_buf.append(scp1644_trim[scp1644_trim.obs.leiden == str(clust_id), :].layers['trimmed_counts'].sum(axis=0))
#     clust_pb_n_cells_buf.append(len(scp1644_trim[scp1644_trim.obs.leiden == str(clust_id), :]))

# adpb = ad.AnnData(pd.DataFrame(clust_pb_buf, index=clust_id_buf, columns=scp1644_trim.var_names))
# adpb.obs['leiden'] = clust_id_buf
# adpb.obs['n_cells'] = np.array(clust_pb_n_cells_buf)
# # print(adpb)

# adpb.var['mt'] = adpb.var_names.str.startswith(("MT-"))
# sc.pp.calculate_qc_metrics(
#     adpb, 
#     qc_vars=["mt"], #, "ribo", "hb"], 
#     inplace=True, 
#     percent_top=[20], 
#     log1p=False,
# )

# adpb.layers['trimmed_counts'] = adpb.to_df().loc[adpb.obs_names, :]
# # sc.pp.scrublet(adpb_all)
# sc.pp.normalize_total(adpb, target_sum=10000, inplace=True)
# sc.pp.log1p(adpb, copy=False)

In [ ]:
for clust_id in sorted(map(int, scp1644_trim.obs.leiden.unique())):
    print(clust_id, Counter(scp1644_trim[scp1644_trim.obs.leiden == str(clust_id), :].obs['Coarse_Cell_Annotations']))

In [ ]:
with pd.option_context('display.width', 200):
    print(adpb_trim.obs.sort_values('pct_counts_mt'))

In [ ]:
# remove cluster 32 for significant mitochondrial content
scp1644_filter = scp1644_trim[[x not in {'32'} for x in scp1644_trim.obs.leiden], :]



In [ ]:
print(scp1644_trim[scp1644_trim.obs['Coarse_Cell_Annotations'] == 'Tumor', :].shape)
print(scp1644_trim[scp1644_trim.obs['Coarse_Cell_Annotations'] != 'Tumor', :].shape)

print(scp1644_filter[scp1644_filter.obs['Coarse_Cell_Annotations'] == 'Tumor', :].shape)
print(scp1644_filter[scp1644_filter.obs['Coarse_Cell_Annotations'] != 'Tumor', :].shape)


In [ ]:
# ## Test doublet filtering based on scrublet. Skip doublet filtering for now

# doublet_cutoff = 0.2

# for clust_id in scp1644_trim[scp1644_trim.obs.doublet_score > doublet_cutoff].obs.leiden.unique():
#     print(clust_id)
#     print(Counter(scp1644_trim[scp1644_trim.obs.leiden==clust_id].obs['Coarse_Cell_Annotations']))
#     print(Counter(scp1644_trim[np.logical_and(scp1644_trim.obs.doublet_score > doublet_cutoff, scp1644_trim.obs.leiden==clust_id)].obs['Coarse_Cell_Annotations']))
#     print()

In [ ]:
## reprocess scp1644_filter

print("PCA and kNN")
sc.pp.pca(scp1644_filter)
sc.pp.neighbors(scp1644_filter, n_neighbors=40, n_pcs=50)


In [ ]:
print("Leiden / tSNE")
sc.tl.leiden(scp1644_filter, resolution=2.0)
sc.tl.umap(scp1644_filter)
sc.tl.tsne(scp1644_filter)

In [ ]:
print("HVG + Rank genes groups")
sc.pp.highly_variable_genes(scp1644_filter, flavor="seurat", n_top_genes=1070)
sc.tl.rank_genes_groups(scp1644_filter, 'leiden')

In [ ]:
## Removal of cluster 32 removes mitochondrial content in PC3
sc.set_figure_params(figsize=(8, 8))

sc.pl.pca_loadings(scp1644_trim, components='1,2,3,4')
sc.pl.pca_loadings(scp1644_filter, components='1,2,3,4')

In [ ]:
sc.pl.pca(scp1644_trim, color='pct_counts_mt')
sc.pl.pca(scp1644_filter, color='pct_counts_mt')

In [ ]:
normal_markers = pd.read_excel(flipcrow.paths.DATA_PATH / "markergenes" / "mmc2.xlsx", header=4, dtype=object)
normal_markers = normal_markers.iloc[:, 2:]
normal_markers = normal_markers.loc[:, ['Gene', 'myAUC', 'avg_diff', 'power', 'avg_logFC', 'pct.1', 'pct.2', 'cell.type']]
normal_markers_dict = {}

# filter datetime oddities found in original data probably caused by entering MARCH1, MARCH11 etc in dataset. This is (possibly?) auto converted to 11-Mar, 1-Mar, etc 
for grpid, grp in normal_markers.groupby('cell.type'):
    normal_markers_dict[grpid] = [grp.loc[idx, 'Gene'] for idx in grp.index if type(grp.loc[idx, 'Gene']) == str and float(grp.loc[idx, 'power']) >= 0.6]

#print(normal_markers_dict)
markers_dict = normal_markers_dict.copy()

# markers_dict['mt_genes'] = [x for x in scp1644_trim.var_names if x.startswith('MT-')]

markers_dict['NET_markers'] = [
    'TTR',
    'CHGB',
]

markers_dict['Tumor_keratins'] = [
    'KRT5',
    'KRT6A',
    'KRT6B',
    'KRT7',
    'KRT8', 
    'KRT10',
    'KRT13',
    'KRT14',
    'KRT15',
    'KRT16',
    'KRT17',
    'KRT18',
    'KRT19',
    'KRT23',
]

moffitt_pdac = {
    'Basal': [
        'VGLL1', 
        'UCA1', 
        'S100A2', 
        'LY6D', 
        'SPRR3',
        'SPRR1B',
        'LEMD1',
        'KRT15',
        'CTSV', # 'CTSL2',
        'DHRS9',
        'AREG',
        'CST6',
        'SERPINB3',
        # 'KRT6C', # not present in var_names
        'KRT6A',
        'FAM83A',
        'SCEL',
        'FGFBP1',
        'KRT7',
        'KRT17',
        'GPR87',
        'TNS4',
        'SLC2A1',
        'ANXA8L2',
    ],
    'Classical': [
        'BTNL8',
        'FAM3D',
        'PRR15L', # 'ATAD4',
        'AGR3',
        'CTSE',
        # 'TMEM238L', # 'LOC400573', not present in genes
        # 'LYZ', # common in T cells
        'TFF2',
        'TFF1',
        'ANXA10',
        'LGALS4',
        'PLA2G10',
        'CEACAM6',
        'VSIG2',
        'TSPAN8',
        'ST6GALNAC1',
        'AGR2',
        'TFF3',
        'CYP3A7',
        'MYO1A',
        'CLRN3',
        'KRT20',
        'CDH17',
        'SPINK4',
        'REG4',
    ],
}

# tumor_keratins = {
#     'Tumor_keratins': [
#         'KRT5',
#         'KRT7',
#         'KRT8', 
#         'KRT10',
#         'KRT13',
#         'KRT14',
#         'KRT17',
#         'KRT18',
#         'KRT19',
#     ]
# }



In [ ]:
sc.pl.heatmap(scp1644_trim, markers_dict, 'leiden', standard_scale='var', figsize=(20,20))

In [ ]:
sc.pl.tsne(scp1644_filter, color='leiden', legend_loc="on data")

In [ ]:
# Make large leiden-based heatmap of the markers dict (Table 2)
sc.tl.dendrogram(scp1644_filter, 'leiden')
markers_dict.update(moffitt_pdac)
# sc.pl.heatmap(scp1644_trim, markers_dict, 'leiden', layer='scale', figsize=(20, 20))
sc.pl.heatmap(scp1644_filter, markers_dict, 'leiden', standard_scale='var', figsize=(20, 20), dendrogram=True)
# sc.pl.heatmap(scp1644_trim, markers_dict, 'leiden', standard_scale='obs', figsize=(20, 20), dendrogram=True, vmin=0, vmax=1)

# sc.pl.heatmap(scp1644_trim, markers_dict, 'leiden', standard_scale='obs', figsize=(20, 20))



In [ ]:
sc.set_figure_params(figsize=(15, 15))

sc.pl.tsne(scp1644_filter, color='leiden', legend_loc="on data", size=10)
sc.pl.umap(scp1644_filter, color='leiden', legend_loc="on data", size=10)

In [ ]:
## "pseudobulk" clusters to simplify clustering

single_cell_data = scp1644_filter

adpb_filter = pseudobulk_leiden(scp1644_filter)

In [ ]:
print(adpb_filter)
for k, v in markers_dict.items():
    sc.tl.score_genes(adpb_filter, v, ctrl_size=100, score_name=f"score_{k}")

In [ ]:
print("Dimensionality reduction and clustering")
sc.pp.pca(adpb_filter)
sc.pp.neighbors(adpb_filter) #, n_neighbors=40, n_pcs=50)
# sc.tl.leiden(adpb)
# sc.tl.umap(adpb_filter)
# sc.tl.tsne(adpb_filter)

In [ ]:
sc.tl.dendrogram(adpb_filter, 'leiden')
sc.pl.heatmap(adpb_filter, markers_dict, 'leiden', standard_scale='var', figsize=(20,20), dendrogram=True)

In [ ]:
print(adpb_filter.obs.loc[:, ['score_Tumor_keratins', 'score_Basal', 'score_Classical']].sort_values('score_Tumor_keratins'))
adpb_filter.obs['tumor_eval'] = (adpb_filter.obs.score_Tumor_keratins > 0.) & ((adpb_filter.obs.score_Basal > 0.) | (adpb_filter.obs.score_Classical > 0.))
# print(adpb_filter.obs.tumor_eval)
for idx in adpb_filter.obs.index:
    print(idx, adpb_filter.obs.loc[idx, 'tumor_eval'], Counter(scp1644_filter[scp1644_filter.obs.leiden==str(idx)].obs['Coarse_Cell_Annotations']))
    

In [ ]:
# print(adpb_filter.obs.loc[:, ['score_T_Cells', 'score_T_Regs', 'score_B_Cells', 'score_Macrophage', 'score_Plasma_cell']].sort_values('score_T_Cells'))
# print()

local_cell_annot = {}
for idx in adpb_filter.obs.index:
    
    # print(adpb_filter.obs.loc[[idx], [x for x in adpb_filter.obs.columns if x.startswith('score')]])
    print(adpb_filter.obs.loc[idx, [x for x in adpb_filter.obs.columns if x.startswith('score')]].T.sort_values())
    print(Counter(scp1644_filter[scp1644_filter.obs.leiden==str(idx)].obs['Coarse_Cell_Annotations']))
    print()
    if adpb_filter.obs.loc[idx, 'tumor_eval']:
        local_cell_annot[idx] = 'Tumor'
    else:
        local_cell_annot[idx] = adpb_filter.obs.loc[idx, [x for x in adpb_filter.obs.columns if x.startswith('score')]].T.sort_values().index[-1].removeprefix('score_')
        


In [ ]:
marker_replace_dict = {
    'T_Cells': 'T_NK',
    'Macrophage': 'Macrophage',
    'Tumor_keratins': 'Tumor',
    'Basal': 'Tumor',
    'Classical': 'Tumor',
    'B_Cells': 'B_Cells',
    'DC': 'DC',
    'Mesenchymal': 'Mesenchymal',
    'Liver_Cell': 'Hepatocyte',
    'Plasma_cell': 'Plasma_cell',
    'T_Regs': 'T_Regs',
    'Endothelial': 'Endothelial',
    'pDC_cell': 'pDC_cell',
    'cp_DC': 'XCR1_DC',
    'NET_markers': 'Tumor',
    'Tumor': 'Tumor',
}

In [ ]:
scp1644_filter.obs['local_cell_annot'] = None
for clust_id in scp1644_filter.obs.leiden.unique():
    try:
        scp1644_filter.obs.loc[scp1644_filter.obs.leiden == clust_id, 'local_cell_annot'] = marker_replace_dict[local_cell_annot[clust_id]]
    except KeyError:
        print(clust_id)
        print(local_cell_annot[clust_id])
        raise

In [ ]:
print(scp1644_filter.obs.local_cell_annot)

In [ ]:
print(sum(scp1644_filter.obs['Coarse_Cell_Annotations'] == scp1644_filter.obs.local_cell_annot))
print(sum(scp1644_filter.obs['Coarse_Cell_Annotations'] == scp1644_filter.obs.local_cell_annot)/len(scp1644_filter))

In [ ]:
sc.pl.rank_genes_groups(scp1644_filter, n=30, sharey=False)

In [ ]:
## Cluster 39 is high in mitochondrial genes
## Removal of cluster 32 removes mitochondrial content in PC3
sc.set_figure_params(figsize=(8, 8))

sc.pl.pca_loadings(scp1644_trim, components='1,2,3,4,5,6,7,8')
sc.pl.pca_loadings(scp1644_filter, components='1,2,3,4,5,6,7,8')

In [ ]:
print(scp1644_filter[scp1644_filter.obs['Coarse_Cell_Annotations'] == 'Tumor'].shape[0]-197)

In [ ]:
filter_savepoint = 'scp1644_filter_savepoint.h5ad'
# scp1644_filter.write_h5ad(str(flipcrow.paths.DATA_PATH / "SCP1644" / filter_savepoint))
scp1644_filter = ad.read_h5ad(str(flipcrow.paths.DATA_PATH / "SCP1644" / filter_savepoint))

In [ ]:
print(scp1644_filter[scp1644_filter.obs['Coarse_Cell_Annotations'] == 'Tumor'].shape[0]-197)

In [ ]:
print(markers_dict.keys())

In [ ]:
#sc.tl.dendrogram(scp1644_filter, 'leiden')
markers_dict.update(moffitt_pdac)

reorg_markers_dict = {x:markers_dict[x] for x in ['Macrophage', 'DC', 'pDC_cell', 'cp_DC', 'T_Cells', 'T_Regs', 'B_Cells', 'Plasma_cell', 'Liver_Cell', 'Endothelial', 'Mesenchymal', 'Tumor_keratins']}
# sc.pl.heatmap(scp1644_trim, markers_dict, 'leiden', layer='scale', figsize=(20, 20))
sc.pl.heatmap(scp1644_filter, reorg_markers_dict, 'local_cell_annot', standard_scale='var', figsize=(20, 20), dendrogram=True)

In [ ]:
# sc.pl.clustermap(scp1644_trim)

In [ ]:
single_cell_data = scp1644_filter

def pseudobulk_leiden_subgroup(single_cell_data: ad.AnnData, subgroup: str = 'donor_ID'):
    # pseudobulk code
    adpb_buf = []
    for clust_id in single_cell_data.obs.leiden.unique():
    
        grpid_buf = []
        pb_counts_buf = []
        pb_n_cells_buf = []
        for grpid, grp in single_cell_data[single_cell_data.obs.leiden == str(clust_id), :].obs.groupby('donor_ID'):
            grpid_buf.append(grpid)
            pb_counts_buf.append(single_cell_data[grp.index, :].layers['trimmed_counts'].sum(axis=0))
            pb_n_cells_buf.append(len(grp.index))
        adpb = ad.AnnData(
            pd.DataFrame(pb_counts_buf, columns=single_cell_data.var_names),
            obs={'n_cells': pb_n_cells_buf},
        )
        adpb.obs['leiden'] = clust_id
        adpb.obs['donor_ID'] = grpid_buf
        
        adpb.obs_names = [f"{x}:{y}" for x, y in zip(np.ones(len(grpid_buf), dtype=int)*int(clust_id), grpid_buf)]
        
        adpb_buf.append(adpb)
    
    adpb_all = ad.concat(adpb_buf)
    
    # mitochondrial genes
    adpb_all.var['mt'] = adpb_all.var_names.str.startswith(("MT-"))
    
    sc.pp.calculate_qc_metrics(
        adpb_all, 
        qc_vars=["mt"], #, "ribo", "hb"], 
        inplace=True, 
        percent_top=[20], 
        log1p=False,
    )
    
    adpb_all.layers['trimmed_counts'] = adpb_all.to_df().loc[adpb_all.obs_names, :]
    # sc.pp.scrublet(adpb_all)
    sc.pp.normalize_total(adpb_all, target_sum=10000, inplace=True)
    sc.pp.log1p(adpb_all, copy=False)

    return adpb_all



In [ ]:
adpb_test = pseudobulk_leiden_subgroup(scp1644_filter)

In [ ]:
print(adpb_test.obs)